# Step 2 — Ask Questions (Retrieval-Augmented Generation)

This notebook uses the database we built in `01_Build_Vector_Database.ipynb` to answer
questions about the book.

**The RAG pattern, in 3 steps, for every question we ask:**
1. **Retrieve** — turn the question into an embedding, and ask ChromaDB for the most similar
   chunks of the book.
2. **Augment** — build a prompt that includes those chunks as "context", followed by the
   question.
3. **Generate** — send that prompt to `gpt-4.1-nano` and let it write an answer grounded in
   the retrieved text.

## 1. Imports and setup

Same API key loading as before, plus connecting to the *existing* ChromaDB database on disk
(we don't rebuild it — `01_Build_Vector_Database.ipynb` already did that).

In [1]:
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../../.env")  # .env lives in the project root
client = OpenAI()

In [2]:
CHROMA_PATH = "../chroma_db"
COLLECTION_NAME = "ml_book"

EMBEDDING_MODEL = "text-embedding-3-small"  # must match the model used in notebook 1
CHAT_MODEL = "gpt-4.1-nano"                  # the model that writes the final answer
TOP_K = 4                                     # how many chunks to retrieve per question

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"Connected to collection '{COLLECTION_NAME}' with {collection.count()} chunks.")

Connected to collection 'ml_book' with 843 chunks.


## 2. Retrieve: find the most relevant chunks for a question

We embed the question with the *same* embedding model used for the chunks, so they live in
the same "meaning space" and can be compared directly.

In [3]:
def retrieve_chunks(question, top_k=TOP_K):
    """Return the top_k chunks most relevant to `question`."""
    question_embedding = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[question],
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    return list(zip(documents, metadatas))

## 3. Augment: build a prompt from the retrieved chunks

We number the chunks as "Source 1", "Source 2", etc., and instruct the model to only use
these sources — this keeps answers grounded in the book instead of the model's general
training knowledge.

In [4]:
def build_prompt(question, retrieved_chunks):
    context_sections = []
    for i, (chunk_text, metadata) in enumerate(retrieved_chunks, start=1):
        context_sections.append(f"Source {i} (page {metadata['page']}):\n{chunk_text}")

    context = "\n\n".join(context_sections)

    prompt = f"""Answer the question using ONLY the sources below.
If the sources don't contain the answer, say you don't know.

{context}

Question: {question}
Answer:"""
    return prompt

## 4. Generate: ask GPT-4.1 nano to write the answer

In [5]:
def ask(question):
    retrieved_chunks = retrieve_chunks(question)
    prompt = build_prompt(question, retrieved_chunks)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant answering questions about a machine learning textbook."},
            {"role": "user", "content": prompt},
        ],
    )

    answer = response.choices[0].message.content

    print("ANSWER:\n")
    print(answer)

    print("\n\nSOURCES USED:")
    for chunk_text, metadata in retrieved_chunks:
        print(f"- page {metadata['page']}")

    return answer

## 5. Try it out!

Run the cell below, then change `question` to anything else about the book and re-run.

In [6]:
question = "What is supervised learning?"
answer = ask(question)

ANSWER:

Supervised learning is a type of machine learning where we aim to predict an outcome based on input data, using a training set composed of input/output pairs. The goal is to build a model that can make accurate predictions on new, unseen data. It often involves human effort to create the training set and is commonly used for tasks such as classification and regression.


SOURCES USED:
- page 102
- page 116
- page 39
- page 87


In [7]:
# Try your own question here
question = "What is the difference between classification and regression?"
answer = ask(question)

ANSWER:

The difference between classification and regression is that classification involves predicting a class label from a predefined list of possibilities, such as determining whether an email is spam or not spam. Regression, on the other hand, involves predicting a continuous value, such as income or yield, which can be any number within a range. The key distinction is that regression deals with outcomes that have a sense of continuity, while classification deals with discrete categories.


SOURCES USED:
- page 39
- page 76
- page 94
- page 40
